<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML/W12/Q2/ASSIGNMNT_W12_Q_2_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TASK 2: DBSCAN MANUAL POINT CLASSIFICATION
# IIT Jodhpur BSc - Google Colab
#
# Parameters:
# Epsilon = 1.9
# MinPoints = 4 (includes the point itself)
#
# Deliverables:
# 1. Distance table for P1, P2, P9, P12
# 2. Classification report for all 12 points
# 3. P6 explanation (maximum 100 words)
# 4. Visualization
# 5. PDF report
# 6. CSV files
# 7. ZIP package with automatic download
# ============================================================

!pip -q install pandas numpy matplotlib reportlab

import os
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, Image as RLImage,
    PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from google.colab import files

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EPSILON = 1.9
MIN_POINTS = 4

OUTPUT_DIR = "/content/DBSCAN_Task_2_Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. ENTER THE COMPLETE DATASET
# ------------------------------------------------------------
# The four coordinates supplied in the assignment are entered.
# Fill in the missing eight coordinates from your assignment.

points = {
    "P1":  (3, 7),
    "P2":  (4, 6),
    "P3":  None,       # ENTER COORDINATES
    "P4":  None,       # ENTER COORDINATES
    "P5":  None,       # ENTER COORDINATES
    "P6":  None,       # ENTER COORDINATES
    "P7":  None,       # ENTER COORDINATES
    "P8":  None,       # ENTER COORDINATES
    "P9":  (3, 3),
    "P10": None,       # ENTER COORDINATES
    "P11": None,       # ENTER COORDINATES
    "P12": (2, 4)
}

# Example:
# "P3": (5, 7)
#
# Replace each None with the actual (x, y) coordinates.

missing = [name for name, coord in points.items() if coord is None]

if missing:
    raise ValueError(
        "Missing coordinates for: " + ", ".join(missing) +
        "\nEnter the actual coordinates in the points dictionary, "
        "then run this cell again."
    )

# Validate coordinates
for name, coord in points.items():
    if (
        not isinstance(coord, (tuple, list))
        or len(coord) != 2
        or not all(isinstance(v, (int, float, np.integer, np.floating)) for v in coord)
    ):
        raise ValueError(f"Invalid coordinates for {name}: {coord}")

point_names = list(points.keys())
coordinates = np.array([points[p] for p in point_names], dtype=float)

# ------------------------------------------------------------
# 3. EUCLIDEAN DISTANCE CALCULATION
# ------------------------------------------------------------

def euclidean_distance(point_a, point_b):
    return math.sqrt(
        (point_a[0] - point_b[0]) ** 2 +
        (point_a[1] - point_b[1]) ** 2
    )

distance_matrix = np.zeros((len(point_names), len(point_names)))

for i in range(len(point_names)):
    for j in range(len(point_names)):
        distance_matrix[i, j] = euclidean_distance(
            coordinates[i], coordinates[j]
        )

distance_df = pd.DataFrame(
    distance_matrix,
    index=point_names,
    columns=point_names
)

# ------------------------------------------------------------
# 4. FIND EPSILON NEIGHBORHOODS
# ------------------------------------------------------------

neighborhoods = {}

for i, point in enumerate(point_names):
    neighbors = [
        point_names[j]
        for j in range(len(point_names))
        if distance_matrix[i, j] <= EPSILON
    ]

    # The point itself is included because its distance is zero.
    neighborhoods[point] = neighbors

neighbor_counts = {
    point: len(neighborhoods[point])
    for point in point_names
}

# ------------------------------------------------------------
# 5. CLASSIFY CORE, BORDER, AND NOISE POINTS
# ------------------------------------------------------------

core_points = {
    point for point in point_names
    if neighbor_counts[point] >= MIN_POINTS
}

border_points = set()
noise_points = set()

for point in point_names:
    if point in core_points:
        continue

    # A non-core point is Border if it lies within epsilon
    # of at least one Core point.
    is_border = any(
        core_point in neighborhoods[point]
        for core_point in core_points
    )

    if is_border:
        border_points.add(point)
    else:
        noise_points.add(point)

classification = {}

for point in point_names:
    if point in core_points:
        classification[point] = "Core"
    elif point in border_points:
        classification[point] = "Border"
    else:
        classification[point] = "Noise"

# ------------------------------------------------------------
# 6. CREATE CLASSIFICATION REPORT
# ------------------------------------------------------------

classification_df = pd.DataFrame({
    "Point": point_names,
    "X": [points[p][0] for p in point_names],
    "Y": [points[p][1] for p in point_names],
    "Neighbors within Epsilon": [
        ", ".join(neighborhoods[p]) for p in point_names
    ],
    "Neighborhood Count": [
        neighbor_counts[p] for p in point_names
    ],
    "Classification": [
        classification[p] for p in point_names
    ]
})

# ------------------------------------------------------------
# 7. CREATE DISTANCE TABLES FOR P1, P2, P9, P12
# ------------------------------------------------------------

selected_points = ["P1", "P2", "P9", "P12"]

selected_distance_rows = []

for point in selected_points:
    i = point_names.index(point)

    for j, other_point in enumerate(point_names):
        distance = distance_matrix[i, j]

        selected_distance_rows.append({
            "Reference Point": point,
            "Compared Point": other_point,
            "Euclidean Distance": round(distance, 4),
            "Within Epsilon (1.9)": (
                "Yes" if distance <= EPSILON else "No"
            )
        })

selected_distance_df = pd.DataFrame(selected_distance_rows)

# Also create a compact distance matrix for the four selected points.
selected_matrix_df = distance_df.loc[selected_points].copy()

# ------------------------------------------------------------
# 8. GENERATE P6 EXPLANATION (MAXIMUM 100 WORDS)
# ------------------------------------------------------------

p6_count = neighbor_counts["P6"]
p6_neighbors = neighborhoods["P6"]
p6_category = classification["P6"]

if p6_category == "Core":
    p6_explanation = (
        f"P6 is classified as Core because it has {p6_count} points "
        f"within the epsilon radius of {EPSILON}, including itself. "
        f"This meets the MinPoints requirement of {MIN_POINTS}. "
        f"Its epsilon neighborhood is: {', '.join(p6_neighbors)}."
    )

elif p6_category == "Border":
    nearby_core_points = [
        p for p in core_points
        if p in neighborhoods["P6"]
    ]

    p6_explanation = (
        f"P6 is classified as Border because it has only {p6_count} "
        f"points in its epsilon neighborhood, which is fewer than "
        f"the required {MIN_POINTS}. However, it lies within epsilon "
        f"distance of a Core point ({', '.join(nearby_core_points)}), "
        f"so it belongs to a cluster."
    )

else:
    p6_explanation = (
        f"P6 is classified as Noise because it has only {p6_count} "
        f"points in its epsilon neighborhood, fewer than the required "
        f"{MIN_POINTS}. It is also not within epsilon distance of any "
        f"Core point, so it cannot be assigned to a cluster."
    )

# ------------------------------------------------------------
# 9. CHECK ASSIGNMENT SUCCESS CRITERIA
# ------------------------------------------------------------

print("\n========== SUCCESS CRITERIA CHECK ==========")

print("P11 Classification:", classification["P11"])
print("P1 Classification:", classification["P1"])

if classification["P11"] == "Core":
    print("PASS: P11 is correctly identified as Core.")
else:
    print("WARNING: P11 is NOT Core. Check the entered coordinates.")

if classification["P1"] == "Border":
    print("PASS: P1 is correctly identified as Border.")
else:
    print("WARNING: P1 is NOT Border. Check the entered coordinates.")

print("\nEpsilon:", EPSILON)
print("MinPoints:", MIN_POINTS)
print("Neighborhood counts include the point itself.")

# ------------------------------------------------------------
# 10. SAVE CSV FILES
# ------------------------------------------------------------

distance_df.to_csv(
    os.path.join(OUTPUT_DIR, "Complete_Distance_Matrix.csv")
)

selected_distance_df.to_csv(
    os.path.join(OUTPUT_DIR, "Selected_Distance_Table.csv"),
    index=False
)

selected_matrix_df.to_csv(
    os.path.join(OUTPUT_DIR, "Selected_Points_Distance_Matrix.csv")
)

classification_df.to_csv(
    os.path.join(OUTPUT_DIR, "DBSCAN_Classification_Report.csv"),
    index=False
)

# ------------------------------------------------------------
# 11. SAVE P6 EXPLANATION
# ------------------------------------------------------------

with open(
    os.path.join(OUTPUT_DIR, "P6_Classification_Explanation.txt"),
    "w"
) as f:
    f.write("P6 CLASSIFICATION EXPLANATION\n")
    f.write("=============================\n\n")
    f.write(p6_explanation)
    f.write(f"\n\nWord count: {len(p6_explanation.split())}\n")

# ------------------------------------------------------------
# 12. CREATE VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(10, 7))

for point in point_names:
    i = point_names.index(point)
    x, y = coordinates[i]

    category = classification[point]

    if category == "Core":
        plt.scatter(
            x, y, s=150, marker="o",
            label="Core" if point == next(iter(core_points), None) else ""
        )
    elif category == "Border":
        plt.scatter(
            x, y, s=150, marker="s",
            label="Border" if point == next(iter(border_points), None) else ""
        )
    else:
        plt.scatter(
            x, y, s=150, marker="x",
            label="Noise" if point == next(iter(noise_points), None) else ""
        )

    plt.annotate(
        point,
        (x, y),
        xytext=(6, 6),
        textcoords="offset points"
    )

plt.title("DBSCAN Point Classification")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.grid(True)
plt.legend()
plt.tight_layout()

visualization_path = os.path.join(
    OUTPUT_DIR, "DBSCAN_Classification_Visualization.png"
)

plt.savefig(visualization_path, dpi=300)
plt.show()
plt.close()

# ------------------------------------------------------------
# 13. CREATE PDF REPORT
# ------------------------------------------------------------

pdf_path = os.path.join(
    OUTPUT_DIR, "DBSCAN_Manual_Point_Classification_Report.pdf"
)

styles = getSampleStyleSheet()

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=landscape(A4),
    rightMargin=30,
    leftMargin=30,
    topMargin=30,
    bottomMargin=30
)

story = []

story.append(Paragraph(
    "TASK 2: DBSCAN MANUAL POINT CLASSIFICATION",
    styles["Title"]
))

story.append(Spacer(1, 10))

story.append(Paragraph(
    "Parameters: Epsilon = 1.9 | MinPoints = 4",
    styles["Heading2"]
))

story.append(Paragraph(
    "The Euclidean distance formula used is: "
    "d(Pi, Pj) = sqrt((xi - xj)^2 + (yi - yj)^2). "
    "A point is Core if its epsilon neighborhood contains at least "
    "four points, including itself. A non-core point is Border if it "
    "is within epsilon distance of a Core point. Otherwise, it is Noise.",
    styles["BodyText"]
))

story.append(Spacer(1, 12))
story.append(Paragraph(
    "1. Distance Table for P1, P2, P9 and P12",
    styles["Heading2"]
))

distance_table_data = [[
    "Reference",
    "Compared Point",
    "Distance",
    "Within Epsilon"
]]

for _, row in selected_distance_df.iterrows():
    distance_table_data.append([
        row["Reference Point"],
        row["Compared Point"],
        f'{row["Euclidean Distance"]:.4f}',
        row["Within Epsilon (1.9)"]
    ])

distance_table = Table(
    distance_table_data,
    repeatRows=1,
    colWidths=[100, 110, 110, 130]
)

distance_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.darkblue),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 5)
]))

story.append(distance_table)
story.append(PageBreak())

story.append(Paragraph(
    "2. Classification Report for All 12 Points",
    styles["Heading2"]
))

classification_table_data = [[
    "Point",
    "Coordinates",
    "Neighborhood Count",
    "Classification"
]]

for _, row in classification_df.iterrows():
    classification_table_data.append([
        row["Point"],
        f'({row["X"]}, {row["Y"]})',
        str(row["Neighborhood Count"]),
        row["Classification"]
    ])

classification_table = Table(
    classification_table_data,
    repeatRows=1,
    colWidths=[100, 130, 160, 130]
)

classification_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.darkblue),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ("TOPPADDING", (0, 0), (-1, -1), 6)
]))

story.append(classification_table)
story.append(Spacer(1, 12))

story.append(Paragraph(
    "3. P6 Classification Explanation",
    styles["Heading2"]
))

story.append(Paragraph(
    p6_explanation,
    styles["BodyText"]
))

story.append(Spacer(1, 12))

story.append(Paragraph(
    f"Word count: {len(p6_explanation.split())} words",
    styles["Italic"]
))

story.append(PageBreak())

story.append(Paragraph(
    "4. DBSCAN Classification Visualization",
    styles["Heading2"]
))

story.append(RLImage(
    visualization_path,
    width=7.5 * inch,
    height=5.0 * inch
))

doc.build(story)

# ------------------------------------------------------------
# 14. CREATE TEXT SUMMARY
# ------------------------------------------------------------

summary_path = os.path.join(
    OUTPUT_DIR, "DBSCAN_Results_Summary.txt"
)

with open(summary_path, "w") as f:
    f.write("TASK 2: DBSCAN MANUAL POINT CLASSIFICATION\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Epsilon: {EPSILON}\n")
    f.write(f"MinPoints: {MIN_POINTS}\n\n")

    f.write("CLASSIFICATION RESULTS\n")
    f.write("-" * 50 + "\n")

    for point in point_names:
        f.write(
            f"{point}: {classification[point]} | "
            f"Neighborhood count: {neighbor_counts[point]} | "
            f"Neighbors: {', '.join(neighborhoods[point])}\n"
        )

    f.write("\nP6 EXPLANATION\n")
    f.write("-" * 50 + "\n")
    f.write(p6_explanation + "\n")

    f.write("\nSUCCESS CRITERIA\n")
    f.write("-" * 50 + "\n")
    f.write(f"P11 is Core: {classification['P11'] == 'Core'}\n")
    f.write(f"P1 is Border: {classification['P1'] == 'Border'}\n")

# ------------------------------------------------------------
# 15. PACKAGE ALL DELIVERABLES INTO ZIP
# ------------------------------------------------------------

zip_path = "/content/Task_2_DBSCAN_Manual_Point_Classification.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(OUTPUT_DIR):
        full_path = os.path.join(OUTPUT_DIR, filename)
        zipf.write(full_path, arcname=filename)

# ------------------------------------------------------------
# 16. DISPLAY FINAL RESULTS
# ------------------------------------------------------------

print("\n========== FINAL CLASSIFICATION ==========")
display(classification_df)

print("\n========== SELECTED DISTANCE TABLE ==========")
display(selected_distance_df)

print("\n========== P6 EXPLANATION ==========")
print(p6_explanation)
print("Word count:", len(p6_explanation.split()))

print("\n========== GENERATED FILES ==========")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("✓", filename)

print("\nZIP package created:")
print(zip_path)

print("\nDownloading ZIP package...")
files.download(zip_path)

ValueError: Missing coordinates for: P3, P4, P5, P6, P7, P8, P10, P11
Enter the actual coordinates in the points dictionary, then run this cell again.